# 🚀 Coffee Chain Demand Forecasting - CatBoost Hybrid (Leakage-Safe)
This notebook combines the robust **Panel ML (Anchor Features) approach** with **Advanced Feature Engineering** (Sin/Cos, Buy1Get1, Event types) to achieve a highly accurate and leakage-free baseline.


In [4]:
import os
import sys
import subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error

try:
    from catboost import CatBoostRegressor, Pool
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "catboost"])
    from catboost import CatBoostRegressor, Pool

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)


## 1. Setup Data Paths


In [5]:
# Find data directory automatically
DATA_DIR_CANDIDATES = [
    Path(os.environ.get("COFFEE_DATA_DIR", "")),
    Path("/content/super-ai-engineer-season-6-coffee-chain-hackathon"),
    Path("/kaggle/input/competitions/super-ai-engineer-season-6-coffee-chain-hackathon"),
    Path(r"c:\Users\CPE KMUTT\Documents\GitHub\superai_engineer_ss6\Level 2\Hackathon 5_Demand Forecasting Coffee Chain Hackathon")
]
DATA_DIR = next((p for p in DATA_DIR_CANDIDATES if (p / "train").exists()), DATA_DIR_CANDIDATES[-1])

OUTPUT_PATH = Path("submission_catboost_hybrid.csv")

TRAIN_START = pd.Timestamp("2023-01-01")
TRAIN_END = pd.Timestamp("2024-10-31")
VALID_START = pd.Timestamp("2024-09-01")

HORIZON_TO_DAYS = {"1d": 1, "7d": 7, "1m": 30}
USE_STOCKOUT_WEIGHTS = True
STOCKOUT_WEIGHT = 0.30
SEED = 42

MODEL_PARAMS = {
    "loss_function": "MAE",
    "eval_metric": "MAE",
    "iterations": 2500,
    "learning_rate": 0.045,
    "depth": 6,
    "l2_leaf_reg": 8,
    "random_seed": SEED,
    "od_type": "Iter",
    "od_wait": 150,
    "verbose": 200,
    "allow_writing_files": False,
    "thread_count": -1,
    # "task_type": "GPU", # Uncomment this on Kaggle/Colab with GPU
}

print("DATA_DIR:", DATA_DIR)


DATA_DIR: /kaggle/input/competitions/super-ai-engineer-season-6-coffee-chain-hackathon


## 2. Load Data


In [6]:
def read_csv(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    return df

train_dir = DATA_DIR / "train"
test_dir = DATA_DIR / "test"

txn = read_csv(train_dir / "TRANSACTION.csv")
order = read_csv(train_dir / "ORDER.csv")
inventory = read_csv(train_dir / "INVENTORY.csv")
promo_train = read_csv(train_dir / "PROMOTION.csv")
event_train = read_csv(train_dir / "LOCAL_EVENT.csv")
date_train = read_csv(train_dir / "DATE_DIM.csv")
store_train = read_csv(train_dir / "STORE.csv")
product_train = read_csv(train_dir / "PRODUCT.csv")

promo_test = read_csv(test_dir / "PROMOTION.csv")
event_test = read_csv(test_dir / "LOCAL_EVENT.csv")
date_test = read_csv(test_dir / "DATE_DIM.csv")

sample = pd.read_csv(DATA_DIR / "sample_submission.csv") if (DATA_DIR / "sample_submission.csv").exists() else pd.read_csv(DATA_DIR / "sample_submission_with_id.csv")

order["date"] = pd.to_datetime(order["date"])
inventory["date"] = pd.to_datetime(inventory["date"])
date_train["date"] = pd.to_datetime(date_train["date"])
date_test["date"] = pd.to_datetime(date_test["date"])

## 3. Build Canonical Target & Full Grid


In [7]:
# Build canonical target
target = (
    txn[["order_id", "product_id", "units_sold"]]
    .merge(order[["order_id", "store_id", "date"]], on="order_id", how="left")
    .merge(product_train[["product_id", "category"]], on="product_id", how="left")
    .groupby(["store_id", "category", "date"], as_index=False)["units_sold"]
    .sum()
)
target["store_id"] = target["store_id"].astype(int)
target["category"] = target["category"].astype(str)
target["date"] = pd.to_datetime(target["date"])
target["units_sold"] = target["units_sold"].astype("float32")

# Create full grid to fill zeroes
stores = sorted(target["store_id"].unique())
categories = sorted(target["category"].unique())
dates = pd.date_range(TRAIN_START, TRAIN_END, freq="D")

grid = pd.MultiIndex.from_product([stores, categories, dates], names=["store_id", "category", "date"]).to_frame(index=False)
full = grid.merge(target, on=["store_id", "category", "date"], how="left")
full["units_sold"] = full["units_sold"].fillna(0).astype("float32")
full = full.sort_values(["store_id", "category", "date"]).reset_index(drop=True)

## 4. Stockout Features for Weights


In [8]:
inv = inventory.copy()
inv = inv.merge(product_train[["product_id", "category"]], on="product_id", how="left")
stockout_cat = (
    inv.groupby(["store_id", "category", "date"], as_index=False)
    .agg(stockout_flag=("is_stockout", "max"), stockout_rate=("is_stockout", "mean"))
)
stockout_cat["category"] = stockout_cat["category"].astype(str)

full = full.merge(stockout_cat, on=["store_id", "category", "date"], how="left")
full["stockout_flag"] = full["stockout_flag"].fillna(0).astype("float32")
full["stockout_rate"] = full["stockout_rate"].fillna(0).astype("float32")

## 5. Anchor Features (Leakage-Safe Lags & Rolling)
These are computed AT the `anchor_date` (which will be the `decision_date`).


In [9]:
def rolling_group(series, keys, window, fn):
    grouped = series.groupby(keys, sort=False)
    if fn == "mean": return grouped.transform(lambda s: s.rolling(window, min_periods=1).mean())
    if fn == "std": return grouped.transform(lambda s: s.rolling(window, min_periods=1).std())
    if fn == "max": return grouped.transform(lambda s: s.rolling(window, min_periods=1).max())
    if fn == "nonzero_rate": return grouped.transform(lambda s: (s > 0).astype("float32").rolling(window, min_periods=1).mean())

df = full.sort_values(["store_id", "category", "date"]).copy()
sales_g = df.groupby(["store_id", "category"], sort=False)["units_sold"]

out = df[["store_id", "category", "date"]].rename(columns={"date": "anchor_date"}).copy()
out["sales_lag0"] = df["units_sold"].astype("float32")
for lag in [7, 14, 21, 28, 35, 56, 84]:
    out[f"sales_lag{lag}"] = sales_g.shift(lag).astype("float32")

for w in [7, 14, 21, 28, 56]:
    out[f"sales_roll_mean_{w}"] = rolling_group(df["units_sold"], [df["store_id"], df["category"]], w, "mean")
    out[f"sales_roll_nonzero_{w}"] = rolling_group(df["units_sold"], [df["store_id"], df["category"]], w, "nonzero_rate")

for w in [28, 56]:
    out[f"sales_roll_std_{w}"] = rolling_group(df["units_sold"], [df["store_id"], df["category"]], w, "std")
    out[f"sales_roll_max_{w}"] = rolling_group(df["units_sold"], [df["store_id"], df["category"]], w, "max")

out["sales_cv_28"] = (out["sales_roll_std_28"] / (out["sales_roll_mean_28"] + 1e-5)).astype("float32")
anchor_features = out

## 6. Advanced Date Features (Sin/Cos)


In [10]:
date_all = pd.concat([date_train, date_test], ignore_index=True).drop_duplicates("date")
date_all["date"] = pd.to_datetime(date_all["date"])

# Standard Calendar
date_all["day_of_week_num"] = date_all["date"].dt.dayofweek.astype("int16")
date_all["day_of_month"] = date_all["date"].dt.day.astype("int16")
date_all["month"] = date_all["date"].dt.month.astype("int16")
date_all["quarter"] = date_all["date"].dt.quarter.astype("int16")
date_all["week_number"] = date_all["date"].dt.isocalendar().week.astype("int16")
date_all["day_of_year"] = date_all["date"].dt.dayofyear.astype("int16")

# Cyclic Transformations (Sin/Cos)
date_all["sin_dow"] = np.sin(2 * np.pi * date_all["day_of_week_num"] / 7.0)
date_all["cos_dow"] = np.cos(2 * np.pi * date_all["day_of_week_num"] / 7.0)
date_all["sin_month"] = np.sin(2 * np.pi * date_all["month"] / 12.0)
date_all["cos_month"] = np.cos(2 * np.pi * date_all["month"] / 12.0)
date_all["sin_doy"] = np.sin(2 * np.pi * date_all["day_of_year"] / 365.25)
date_all["cos_doy"] = np.cos(2 * np.pi * date_all["day_of_year"] / 365.25)

# Flags
date_all["is_month_start"] = date_all["date"].dt.is_month_start.astype("int8")
date_all["is_month_end"] = date_all["date"].dt.is_month_end.astype("int8")
date_all["is_songkran_window"] = ((date_all["month"] == 4) & date_all["day_of_month"].between(10, 17)).astype("int8")

bool_cols = ["is_weekend", "is_holiday", "is_school_break", "is_payday", "is_rainy_season"]
for c in bool_cols:
    date_all[c] = date_all.get(c, 0).fillna(0).astype(int)

date_features = date_all

## 7. Advanced Event & Promo Features


In [12]:
# EVENT FEATURES
event = pd.concat([event_train, event_test], ignore_index=True).drop_duplicates()
event["date"] = pd.to_datetime(event["date"])
event_agg = event.groupby(["store_id", "date"]).agg(
    event_count=("event_id", "nunique"),
    event_types=("event_type", lambda x: " ".join(x.dropna().astype(str).str.lower()))
).reset_index().rename(columns={"date": "forecast_date"})

event_agg["is_food_festival"] = event_agg["event_types"].str.contains("food_festival|food festival").astype(int)
event_agg["is_concert"] = event_agg["event_types"].str.contains("concert").astype(int)
event_agg["is_sports"] = event_agg["event_types"].str.contains("sport").astype(int)
event_features = event_agg.drop(columns=["event_types"])

# PROMO FEATURES
promo = pd.concat([promo_train, promo_test], ignore_index=True).drop_duplicates()
promo = promo.merge(product_train[["product_id", "category"]], on="product_id", how="left")
promo["start_date"] = pd.to_datetime(promo["start_date"])
promo["end_date"] = pd.to_datetime(promo["end_date"])

# Safely join text columns if they exist
text_cols = [c for c in ["campaign", "campaign_type", "campaign_name", "campaign_id"] if c in promo.columns]
if text_cols:
    promo_text = promo[text_cols[0]].fillna("").astype(str)
    for c in text_cols[1:]:
        promo_text = promo_text + " " + promo[c].fillna("").astype(str)
    promo["promo_text"] = promo_text.str.lower()
else:
    promo["promo_text"] = ""

promo["is_buy1get1"] = promo["promo_text"].str.contains("1แถม1|buy1get1|b1g1|1 แถม 1").astype(int)

promo_rows = []
for _, r in promo.iterrows():
    if pd.isna(r["start_date"]) or pd.isna(r["category"]): continue
    start = max(pd.Timestamp(r["start_date"]), TRAIN_START)
    end = min(pd.Timestamp(r["end_date"]), pd.Timestamp("2024-12-31"))
    if end < start: continue
    store_values = stores if pd.isna(r.get("store_id")) else [int(r["store_id"])]
    
    for store_id in store_values:
        for d in pd.date_range(start, end, freq="D"):
            promo_rows.append({
                "store_id": store_id, "category": str(r["category"]), "forecast_date": d,
                "promo_count": 1, 
                "discount_pct": float(0 if pd.isna(r.get("discount_pct")) else r["discount_pct"]),
                "is_buy1get1": r["is_buy1get1"]
            })

p_df = pd.DataFrame(promo_rows)
if len(p_df) > 0:
    promo_features = p_df.groupby(["store_id", "category", "forecast_date"], as_index=False).agg(
        promo_count=("promo_count", "sum"),
        promo_discount_max=("discount_pct", "max"),
        promo_buy1get1=("is_buy1get1", "max")
    )
else:
    promo_features = pd.DataFrame(columns=["store_id", "category", "forecast_date", "promo_count", "promo_discount_max", "promo_buy1get1"])


## 8. Store & Category Meta


In [13]:
store_train["opened_date"] = pd.to_datetime(store_train["opened_date"])
store_train["neighborhood_type"] = store_train.get("neighborhood_type", "missing").fillna("missing").astype(str)
store_features = store_train

cat_features = product_train.groupby("category", as_index=False).agg(
    product_count=("product_id", "nunique"),
    avg_base_price=("base_price", "mean")
)


## 9. Create Train/Test Frames


In [14]:
def parse_submission(sample):
    out = sample.copy()
    parsed = out["id"].str.extract(r"^(\d+)_(.+)_(\d{4}-\d{2}-\d{2})_(1d|7d|1m)$")
    out[["store_id", "category", "forecast_date", "horizon"]] = parsed
    out["store_id"] = out["store_id"].astype(int)
    out["category"] = out["category"].astype(str)
    out["forecast_date"] = pd.to_datetime(out["forecast_date"])
    out["h"] = out["horizon"].map(HORIZON_TO_DAYS).astype(int)
    out["decision_date"] = out["forecast_date"] - pd.to_timedelta(out["h"], unit="D")
    return out

parsed_sample = parse_submission(sample)

def add_all_features(df):
    out = df.copy()
    out = out.merge(date_features.rename(columns={"date": "forecast_date"}), on="forecast_date", how="left")
    out = out.merge(store_features, on="store_id", how="left")
    out = out.merge(cat_features, on="category", how="left")
    out = out.merge(promo_features, on=["store_id", "category", "forecast_date"], how="left")
    out = out.merge(event_features, on=["store_id", "forecast_date"], how="left")
    
    out["store_age_days"] = (out["forecast_date"] - out["opened_date"]).dt.days.astype("float32")
    
    for c in ["promo_count", "promo_discount_max", "promo_buy1get1", "event_count", "is_food_festival", "is_concert", "is_sports"]:
        if c in out.columns: out[c] = out[c].fillna(0).astype("float32")
    return out

# Train Frame
train_frames = []
for horizon, h in HORIZON_TO_DAYS.items():
    base = full[["store_id", "category", "date", "units_sold", "stockout_flag", "stockout_rate"]].rename(
        columns={"date": "forecast_date", "units_sold": "target_units_sold"}
    )
    base["horizon"] = horizon
    base["h"] = h
    base["decision_date"] = base["forecast_date"] - pd.to_timedelta(h, unit="D")
    base["anchor_date"] = base["decision_date"]
    base = base[base["anchor_date"] >= TRAIN_START].copy()
    
    base = base.merge(anchor_features, on=["store_id", "category", "anchor_date"], how="left")
    base = add_all_features(base)
    
    rate = base["stockout_rate"].fillna(0).clip(0, 1)
    base["sample_weight"] = (1.0 - (1.0 - STOCKOUT_WEIGHT) * rate).astype("float32")
    train_frames.append(base)

train_all = pd.concat(train_frames, ignore_index=True)

# Test Frame
test_all = parsed_sample.copy()
test_all["anchor_date"] = test_all["decision_date"].clip(upper=TRAIN_END)
test_all = test_all.merge(anchor_features, on=["store_id", "category", "anchor_date"], how="left")
test_all = add_all_features(test_all)


## 10. Training (CatBoost)


In [15]:
feature_cols = [
    "store_id", "category",
    "sales_lag0", "sales_lag7", "sales_lag14", "sales_lag21", "sales_lag28", "sales_lag35", "sales_lag56", "sales_lag84",
    "sales_roll_mean_7", "sales_roll_mean_14", "sales_roll_mean_21", "sales_roll_mean_28", "sales_roll_mean_56",
    "sales_roll_std_28", "sales_roll_max_28", "sales_roll_std_56", "sales_roll_max_56",
    "sales_roll_nonzero_7", "sales_roll_nonzero_14", "sales_roll_nonzero_21", "sales_roll_nonzero_28", "sales_roll_nonzero_56",
    "sales_cv_28",
    "day_of_week_num", "is_weekend", "is_holiday", "is_school_break", "is_payday", "is_rainy_season", 
    "month", "quarter", "day_of_month", "week_number", "is_month_start", "is_month_end", "is_songkran_window",
    "sin_dow", "cos_dow", "sin_month", "cos_month", "sin_doy", "cos_doy",
    "neighborhood_type", "store_age_days", "product_count", "avg_base_price",
    "promo_count", "promo_discount_max", "promo_buy1get1",
    "event_count", "is_food_festival", "is_concert", "is_sports"
]
cat_cols = ["store_id", "category", "neighborhood_type"]

def clean_features(df):
    X = df[feature_cols].copy()
    for c in cat_cols: X[c] = X[c].fillna("missing").astype(str)
    for c in feature_cols:
        if c not in cat_cols: X[c] = pd.to_numeric(X[c], errors="coerce").astype("float32")
    return X

models = {}
for horizon in ["1d", "7d", "1m"]:
    print(f"\n=== Training {horizon} ===")
    df_h = train_all[train_all["horizon"] == horizon].copy()
    
    # Train on Full data directly for best Kaggle score (no validation holdout for final sub)
    X_full = clean_features(df_h)
    y_full = df_h["target_units_sold"].astype("float32")
    w_full = df_h["sample_weight"].values if USE_STOCKOUT_WEIGHTS else None
    
    cat_idx = [X_full.columns.get_loc(c) for c in cat_cols]
    pool = Pool(X_full, y_full, cat_features=cat_idx, weight=w_full)
    
    model = CatBoostRegressor(**MODEL_PARAMS)
    model.fit(pool)
    models[horizon] = model



=== Training 1d ===
0:	learn: 29.6161548	total: 115ms	remaining: 4m 46s
200:	learn: 9.5300596	total: 9.07s	remaining: 1m 43s
400:	learn: 8.8073580	total: 17.9s	remaining: 1m 33s
600:	learn: 8.4725233	total: 26.9s	remaining: 1m 24s
800:	learn: 8.2549978	total: 35.9s	remaining: 1m 16s
1000:	learn: 8.0879004	total: 44.9s	remaining: 1m 7s
1200:	learn: 7.9522950	total: 53.9s	remaining: 58.3s
1400:	learn: 7.8353100	total: 1m 2s	remaining: 49.4s
1600:	learn: 7.7372791	total: 1m 11s	remaining: 40.4s
1800:	learn: 7.6339258	total: 1m 20s	remaining: 31.4s
2000:	learn: 7.5492855	total: 1m 29s	remaining: 22.4s
2200:	learn: 7.4723876	total: 1m 38s	remaining: 13.4s
2400:	learn: 7.4045998	total: 1m 48s	remaining: 4.45s
2499:	learn: 7.3670450	total: 1m 52s	remaining: 0us

=== Training 7d ===
0:	learn: 29.6950265	total: 49.8ms	remaining: 2m 4s
200:	learn: 9.4996627	total: 8.96s	remaining: 1m 42s
400:	learn: 8.8022538	total: 17.8s	remaining: 1m 33s
600:	learn: 8.4701342	total: 26.8s	remaining: 1m 24s
80

## 11. Inference


In [16]:
submission = sample[["id"]].copy()
submission["units_sold_predicted"] = np.nan

for horizon in ["1d", "7d", "1m"]:
    idx = test_all["horizon"] == horizon
    X_test = clean_features(test_all.loc[idx])
    pred = np.maximum(models[horizon].predict(X_test), 0)
    submission.loc[idx.values, "units_sold_predicted"] = pred

submission.to_csv(OUTPUT_PATH, index=False)
print("Saved to", OUTPUT_PATH)
display(submission.head())

Saved to submission_catboost_hybrid.csv


,id,units_sold_predicted
0,1_Bakery_2024-11-01_1d,36.981098
1,1_Bakery_2024-11-01_1m,33.456344
2,1_Bakery_2024-11-01_7d,37.615348
3,1_Bakery_2024-11-02_1d,15.584649
4,1_Bakery_2024-11-02_1m,13.965257
